In [1]:
# Parameters
BATCH_MODE = "true"


# SC-13-Fuzz-Invariants - Fuzz Testing

[<< Precedent : Foundry Testing](SC-12-Foundry-Testing.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Formal Verification >>](SC-14-Formal-Verification.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre le **fuzz testing** et ses avantages
2. Ecrire des tests avec **parametres aleatoires**
3. Tester des **invariants** de contrats
4. Utiliser `vm.assume` pour filtrer les entrees

### Prerequis

- [SC-12-Foundry-Testing](SC-12-Foundry-Testing.ipynb) complete
- Foundry installe et projet initialise
- Notions de tests unitaires en Solidity

### Duree estimee : 40 minutes

---

## 1. Introduction au Fuzz Testing

Le fuzz testing genere automatiquement des entrees aleatoires pour trouver des bugs.

In [2]:
# Concept de fuzz testing
print("""
FUZZ TESTING - Concepts cles

1. GENERATION ALEATOIRE
   - Foundry genere des valeurs aleatoires pour chaque parametre
   - Types supportes: uint, int, address, bytes, string, arrays

2. CAS D'USAGE
   - Trouver des edge cases non couverts par les tests unitaires
   - Tester des bornes (overflow, underflow)
   - Verifier des invariants mathematiques

3. CONFIGURATION
   - runs: nombre d'executions (defaut: 256)
   - depth: profondeur des appels internes
   - seed: pour reproduire un bug specifique

4. COMMANDE
   forge test --fuzz-runs 1000 --match-path test/Fuzz
""")


FUZZ TESTING - Concepts cles

1. GENERATION ALEATOIRE
   - Foundry genere des valeurs aleatoires pour chaque parametre
   - Types supportes: uint, int, address, bytes, string, arrays

2. CAS D'USAGE
   - Trouver des edge cases non couverts par les tests unitaires
   - Tester des bornes (overflow, underflow)
   - Verifier des invariants mathematiques

3. CONFIGURATION
   - runs: nombre d'executions (defaut: 256)
   - depth: profondeur des appels internes
   - seed: pour reproduire un bug specifique

4. COMMANDE
   forge test --fuzz-runs 1000 --match-path test/Fuzz



---

## 2. Fuzz Test Simple

In [3]:
# Fuzz test basique
FUZZ_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SafeMath {
    // Fonction a tester
    function safeAdd(uint256 a, uint256 b) public pure returns (uint256) {
        uint256 c = a + b;
        require(c >= a, "Overflow");
        return c;
    }
    
    function safeMul(uint256 a, uint256 b) public pure returns (uint256) {
        if (a == 0) return 0;
        uint256 c = a * b;
        require(c / a == b, "Overflow");
        return c;
    }
}
'''

FUZZ_TEST_BASIC = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SafeMath.sol";

contract SafeMathFuzzTest is Test {
    SafeMath sm;

    function setUp() public {
        sm = new SafeMath();
    }

    // Fuzz test: a et b sont generes aleatoirement
    function testFuzz_SafeAdd(uint256 a, uint256 b) public pure {
        // Si pas d'overflow, safeAdd doit reussir
        if (a + b >= a) {
            assertEq(sm.safeAdd(a, b), a + b);
        }
    }

    // Fuzz test avec contrainte
    function testFuzz_SafeMul(uint256 a, uint256 b) public pure {
        vm.assume(a > 0);  // Eviter division par zero
        vm.assume(b > 0);  // Cas non-trivial
        
        uint256 result = sm.safeMul(a, b);
        assertEq(result, a * b);
    }

    // Test avec bounds explicites
    function testFuzz_SafeAdd_Bounds() public pure {
        // Tester avec max uint256
        vm.assume(uint128(a) + uint128(b) == uint128(a) + uint128(b));
        assertEq(sm.safeAdd(a, b), a + b);
    }
}
'''

print("Contrat SafeMath:")
print(FUZZ_BASIC)
print("\n" + "="*50 + "\n")
print("Test Fuzz:")
print(FUZZ_TEST_BASIC)

Contrat SafeMath:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract SafeMath {
    // Fonction a tester
    function safeAdd(uint256 a, uint256 b) public pure returns (uint256) {
        uint256 c = a + b;
        require(c >= a, "Overflow");
        return c;
    }

    function safeMul(uint256 a, uint256 b) public pure returns (uint256) {
        if (a == 0) return 0;
        uint256 c = a * b;
        require(c / a == b, "Overflow");
        return c;
    }
}



Test Fuzz:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SafeMath.sol";

contract SafeMathFuzzTest is Test {
    SafeMath sm;

    function setUp() public {
        sm = new SafeMath();
    }

    // Fuzz test: a et b sont generes aleatoirement
    function testFuzz_SafeAdd(uint256 a, uint256 b) public pure {
        // Si pas d'overflow, safeAdd doit reussir
        if (a + b >= a) {
            assertEq(sm.safeAdd(a, b), a + b);
        }
    }

   

### Interpretation : Concepts du Fuzz Testing

Cette synthese presente les principes fondamentaux du fuzz testing avec Foundry.

| Concept | Description | Impact sur les tests |
|---------|-------------|---------------------|
| **Generation aleatoire** | Foundry genere des valeurs pour tous les parametres `uint`/`int`/`address`/`bytes`/`arrays` | Couvre des cas auxquels le developpeur n'a pas pense |
| **Edge cases** | Bornes extremes (0, max uint256, overflow, underflow) | Trouve des bugs de debordement arithmetique |
| **Invariants mathematiques** | Proprietes qui doivent toujours etre vraies (ex: somme des balances = total supply) | Valide la coherence de l'etat du contrat |
| **Configuration `runs`** | Nombre d'iterations par test (defaut 256) | Plus de runs = plus de couverture, mais plus lent |

**Points cles** :
1. Le fuzz testing est complementaire aux tests unitaires : il explore l'espace des inputs de maniere systematique et aleatoire
2. `depth` dans la configuration correspond a la profondeur des appels de fonctions imbriques (pour les invariants)
3. Le parametre `seed` permet la reproductibilite : meme bug, meme seed = meme echec
4. La commande `forge test --fuzz-runs 1000` augmente la couverture pour des tests plus exhaustifs

**Cas d'usage typiques** :
- Tester des fonctions avec des parametres utilisateurs non controlees
- Valider des contraintes d'access control (roles, permissions)
- Verifier la coherence d'invariants comptables (balances, supply, ratios)

> **Note technique** : Le fuzz testing avec Foundry est "stateless" par defaut (chaque test repart de zero). Pour du "stateful fuzzing" (plusieurs transactions chainees), il faut utiliser la section `[invariant]` de foundry.toml.


---

## 3. Invariant Testing

Les invariants testent que des proprietes restent toujours vraies.

In [4]:
# Invariant testing pour un token ERC-20
INVARIANT_TEST = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SimpleToken.sol";

contract TokenInvariantTest is Test {
    SimpleToken token;
    address[] users;

    function setUp() public {
        token = new SimpleToken(1_000_000 * 10**18);
        
        // Creer des utilisateurs
        for (uint i = 0; i < 5; i++) {
            users.push(address(uint160(i + 1)));
        }
    }

    // Invariant: le total supply ne change jamais
    function invariant_TotalSupply() public view {
        uint256 initialSupply = token.totalSupply();
        assertEq(token.totalSupply(), initialSupply);
    }

    // Invariant: sum of balances == total supply
    function invariant_BalanceSum() public view {
        uint256 sum = 0;
        for (uint i = 0; i < users.length; i++) {
            sum += token.balanceOf(users[i]);
        }
        // Note: peut echouer si mint/burn existe
    }
}
'''

print("Test d'Invariants:")
print(INVARIANT_TEST)

Test d'Invariants:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "forge-std/Test.sol";
import "../src/SimpleToken.sol";

contract TokenInvariantTest is Test {
    SimpleToken token;
    address[] users;

    function setUp() public {
        token = new SimpleToken(1_000_000 * 10**18);

        // Creer des utilisateurs
        for (uint i = 0; i < 5; i++) {
            users.push(address(uint160(i + 1)));
        }
    }

    // Invariant: le total supply ne change jamais
    function invariant_TotalSupply() public view {
        uint256 initialSupply = token.totalSupply();
        assertEq(token.totalSupply(), initialSupply);
    }

    // Invariant: sum of balances == total supply
    function invariant_BalanceSum() public view {
        uint256 sum = 0;
        for (uint i = 0; i < users.length; i++) {
            sum += token.balanceOf(users[i]);
        }
        // Note: peut echouer si mint/burn existe
    }
}



### Interpretation : Fuzz Tests pour Operations Arithmetiques Securisees

Les tests presentent des strategies de fuzzing pour valider les protections contre overflow/underflow.

| Test | Strategie | Validation |
|------|-----------|------------|
| `testFuzz_SafeAdd` | Cas general | Verifie que le resultat correspond a l'addition native si pas d'overflow |
| `testFuzz_SafeMul` | Filtrage inputs | `vm.assume` evite les cas triviaux (a=0 ou b=0) |
| `testFuzz_SafeAdd_Bounds` | Bornes explicites | Teste uniquement le cas uint128 (pas d'overflow possible) |

**Points cles** :
1. `testFuzz_SafeAdd` utilise une condition `if (a + b >= a)` pour detecter l'overflow AVANT d'appeler la fonction
2. `testFuzz_SafeMul` combine `vm.assume(a > 0)` et `vm.assume(b > 0)` pour eviter les cas edge (division par zero dans la verification)
3. Le pattern `assertEq(result, a * b)` verifie que le resultat correspond a l'operation native (qui peut overflow en Solidity 0.8+)
4. `testFuzz_SafeAdd_Bounds` montre une approche conservative : limiter les inputs a une plage safe (uint128)

**Difference fondamentale** : Solidity 0.8+ a des checks d'overflow automatiques (`0x...` revert), mais les tests fuzz valident que la logique de protection (`require`) fonctionne correctement.

> **Note technique** : Le fuzzing trouve des bugs que les tests unitaires manquent. Par exemple, un test unitaire pourrait tester `1 + 1 = 2`, mais le fuzz testera `2^256 - 1 + 1` et detectera l'overflow.


---

## 4. vm.assume vs require

| Instruction | Usage | Effet |
|-------------|------|------|
| `vm.assume(cond)` | Dans les fuzz tests | Rejette l'input et genere une nouvelle |
| `require(cond, msg)` | Dans le contrat | Revert la transaction (echec du test) |

In [5]:
# Exemple avec vm.assume
ASSUME_EXAMPLE = '''
contract AssumeExample is Test {
    // Fuzz test avec filtrage d'inputs
    function testFuzz_Filtered(
        uint256 a,
        uint256 b,
        address addr
    ) public pure {
        // Filtrer les cas non desires
        vm.assume(a > 0);           // a non nul
        vm.assume(b > 0);           // b non nul
        vm.assume(a < type(uint128).max);  // a dans les limites
        vm.assume(addr != address(0));  // adresse non nulle
        vm.assume(addr.code.length == 0);  // pas un contrat
        
        // Maintenant tester avec des entrees valides
        uint256 result = a + b;
        assertGt(result, a);
        assertGt(result, b);
    }

    // Pattern commun de filtrage
    function testFuzz_AddressFilter(address addr) public pure {
        vm.assume(addr != address(0));
        vm.assume(addr != 0x0000000000000000000000000000000000000001);  // Ecrou
        // ... test logic
    }

    function testFuzz_UintRange(uint256 x) public pure {
        vm.assume(x >= 100 && x <= 1000);
        // x est maintenant dans [100, 1000]
    }
}
'''

print("Patterns avec vm.assume:")
print(ASSUME_EXAMPLE)

Patterns avec vm.assume:

contract AssumeExample is Test {
    // Fuzz test avec filtrage d'inputs
    function testFuzz_Filtered(
        uint256 a,
        uint256 b,
        address addr
    ) public pure {
        // Filtrer les cas non desires
        vm.assume(a > 0);           // a non nul
        vm.assume(b > 0);           // b non nul
        vm.assume(a < type(uint128).max);  // a dans les limites
        vm.assume(addr != address(0));  // adresse non nulle
        vm.assume(addr.code.length == 0);  // pas un contrat

        // Maintenant tester avec des entrees valides
        uint256 result = a + b;
        assertGt(result, a);
        assertGt(result, b);
    }

    // Pattern commun de filtrage
    function testFuzz_AddressFilter(address addr) public pure {
        vm.assume(addr != address(0));
        vm.assume(addr != 0x0000000000000000000000000000000000000001);  // Ecrou
        // ... test logic
    }

    function testFuzz_UintRange(uint256 x) public pure {
      

### Interpretation : Tests d'Invariants pour Token ERC-20

Ce code presente des tests de proprietes qui doivent toujours etre vraies pour un token ERC-20.

| Invariant | Propriete testee | Implementation |
|-----------|-----------------|----------------|
| `invariant_TotalSupply` | Le supply total est constant | Verifie que `totalSupply()` ne change jamais |
| `invariant_BalanceSum` | Somme des balances = supply | Additionne tous les balances et compare au supply |

**Points cles** :
1. Les fonctions `invariant_*` sont appelees automatiquement par Foundry entre chaque operation aleatoire
2. `invariant_TotalSupply` est valable uniquement s'il n'y a pas de fonctions `mint`/`burn`
3. `invariant_BalanceSum` est l'invariant fondamental d'ERC-20 : conservation de la masse monetaire
4. La boucle `for` dans `invariant_BalanceSum` ne couvre que 5 utilisateurs predefinis (pas exhaustif)

**Limitation** : Le test note que `invariant_BalanceSum` peut echouer si des fonctions `mint`/`burn` existent, car elles brisent l'invariant de conservation (ce qui est normal pour ces operations).

> **Note technique** : Foundry execute les invariants en appelant des fonctions du contrat de maniere aleatoire (fuzzing oriente etat), puis verifie que les invariants restent valides. La `depth` dans foundry.toml controle combien d'appels sont chaines.


---

## 5. Configuration Fuzz

Dans `foundry.toml`:

In [6]:
# Configuration foundry.toml
FOUNDRY_CONFIG = '''
[fuzz]
runs = 1000          # Nombre d'executions par test
max_test_rejects = 1000  # Max rejets avant abandon
seed = '0x1234'    # Seed pour reproductibilite
dictionary_weight = 40  # Poids du dictionnaire

[invariant]
runs = 256           # Runs par invariant
depth = 15           # Profondeur des appels
fail_on_revert = true  # Echouer sur revert
'''

# Commandes
COMMANDS = '''
# Executer les fuzz tests
forge test --fuzz-runs 5000

# Executer uniquement les tests fuzz
forge test --match-path "test/*Fuzz*"

# Avec un seed specifique
forge test --fuzz-seed 0x3e8e

# Debug mode
forge test -vvvv --fuzz-runs 10
'''

print("Configuration foundry.toml:")
print(FOUNDRY_CONFIG)
print("\n" + "="*50 + "\n")
print("Commandes:")
print(COMMANDS)

Configuration foundry.toml:

[fuzz]
runs = 1000          # Nombre d'executions par test
max_test_rejects = 1000  # Max rejets avant abandon
seed = '0x1234'    # Seed pour reproductibilite
dictionary_weight = 40  # Poids du dictionnaire

[invariant]
runs = 256           # Runs par invariant
depth = 15           # Profondeur des appels
fail_on_revert = true  # Echouer sur revert



Commandes:

# Executer les fuzz tests
forge test --fuzz-runs 5000

# Executer uniquement les tests fuzz
forge test --match-path "test/*Fuzz*"

# Avec un seed specifique
forge test --fuzz-seed 0x3e8e

# Debug mode
forge test -vvvv --fuzz-runs 10



### Exercice 3 : Invariant d'une PriorityQueue — le parent est toujours >= aux enfants

Ecrivez un test d'invariant qui verifie que dans une file de priorite (max-heap),
chaque parent est toujours superieur ou egal a ses enfants.

**Objectifs** :
1. Implementer `checkHeapProperty()` qui parcourt le tableau
2. Verifier `heap[i] >= heap[2*i+1]` et `heap[i] >= heap[2*i+2]` pour tout i valide
3. Generer des sequences d'insertions aleatoires pour tester

**Indice** : En Solidity, l'invariant se teste avec `for (uint i = 0; 2*i+1 < length; i++)`.


In [7]:
# Exercice 3 : Invariant d'une PriorityQueue (max-heap)
# TODO etudiant : ecrire un invariant qui verifie la propriete de tas
# Etape 1 : Implementer checkHeapProperty(heap) en Python
# Etape 2 : Tester avec des insertions aleatoires
# Etape 3 : Verifier que l'invariant tient toujours
import random

def check_heap_property(heap):
    # TODO etudiant : retourner True si heap[i] >= heap[2*i+1] et heap[2*i+2] pour tout i
    return True  # TODO etudiant

# heap = [100, 90, 80, 70, 60, 50]
# print("Heap valide:", check_heap_property(heap))
print("Exercice a completer")


Exercice a completer


### Exercice 2 : Fuzz test d'une fonction de transfert ERC-20

Ecrivez un fuzz test en Python qui verifie qu'un transfert ERC-20 ne cree ni ne
detruit de tokens (conservation du totalSupply).

**Objectifs** :
1. Implementer `fuzz_transfer(total_supply, sender_balance, amount)` en Python
2. Verifier que `total_supply` reste constant apres transfer
3. Verifier que les soldes individuels restent >= 0

**Indice** : `sender_balance_after = sender_balance - amount` et `receiver_balance_after = receiver_balance + amount`. Total = constante.


In [8]:
# Exercice 2 : Fuzz test de conservation du totalSupply ERC-20
# TODO etudiant : tester que le totalSupply ne change pas apres des transferts
# Etape 1 : Definir fuzz_transfer(total_supply, n_transfers) avec des montants aleatoires
# Etape 2 : Verifier sum(balances) == total_supply apres chaque transfert
# Etape 3 : Verifier balance >= 0 apres chaque transfert
import random

def fuzz_transfer(total_supply, n_transfers=100):
    # TODO etudiant : simuler n_transfers et verifier la conservation
    return True  # TODO etudiant

# print("Conservation:", fuzz_transfer(1000000, 100))
print("Exercice a completer")


Exercice a completer


### Interpretation : Patterns vm.assume

Les exemples montrent comment filtrer efficacement les inputs aleatoires pour se concentrer sur les cas interessants.

| Pattern | Usage | Exemple |
|---------|-------|----------|
| `vm.assume(x > 0)` | Exclure zero | Eviter division par zero |
| `vm.assume(x < bound)` | Borner les valeurs | Garder dans uint128 pour eviter overflow |
| `vm.assume(addr.code.length == 0)` | EOA only | Exclure les contrats intelligents |
| `vm.assume(x >= 100 && x <= 1000)` | Range specifique | Tester uniquement une plage pertinente |

**Points cles** :
1. `vm.assume` ne fait pas echouer le test : elle rejette l'input et en genere un nouveau
2. Trop de `vm.assume` ralentit le fuzz (beaucoup d'inputs rejetes)
3. L'adresse `0x0000000000000000000000000000000000000001` est un "ecrou" (precompile) souvent exclue
4. `addr.code.length` distingue EOA (0) de contrats (>0) : utile pour les fonctions `payable`

> **Note technique** : La difference fondamentale : `vm.assume` filtre les inputs AVANT execution (fuzzing), `require` fait echouer le test PENDANT execution (contract logic).


---

## 6. Exemple guide et Exercice

L'exemple guidé ci-dessous montre un fuzz test résolu (solution étudiante). Un exercice à compléter suit.

In [9]:
# Exemple guide : Fuzz tests pour une pile LIFO (solution Mehdi Robardet + Oceane Xiang, TP 2026)
# Contrat Stack + fuzz tests testFuzz_PushPop et testFuzz_LIFO
# Solution etudiante : Mehdi Robardet, Oceane Xiang (TP 2026)

EXERCISE_STACK = '''
// Contrat a tester
contract Stack {
    uint256[] private items;
    
    function push(uint256 item) public {
        items.push(item);
    }
    
    function pop() public returns (uint256) {
        require(items.length > 0, "Empty stack");
        return items.pop();
    }
    
    function peek() public view returns (uint256) {
        require(items.length > 0, "Empty stack");
        return items[items.length - 1];
    }
    
    function size() public view returns (uint256) {
        return items.length;
    }
}

// Fuzz test
contract StackFuzzTest is Test {
    Stack stack;
    
    function setUp() public {
        stack = new Stack();
    }
    
    // Invariant: apres push puis pop, size est inchange
    function testFuzz_PushPop(uint256 value) public {
        uint256 sizeBefore = stack.size();

        stack.push(value);
        assertEq(stack.size(), sizeBefore + 1);

        uint256 popped = stack.pop();
        assertEq(popped, value);

        assertEq(stack.size(), sizeBefore);
    }
    
    // Invariant: LIFO property
    function testFuzz_LIFO(uint256[] memory values) public {
        vm.assume(values.length > 0 && values.length <= 100);

        for (uint i = 0; i < values.length; i++) {
            stack.push(values[i]);
        }

        for (uint i = values.length; i > 0; i--) {
            assertEq(stack.pop(), values[i - 1]);
        }
    }
}
'''
print("Exemple guide : Stack fuzz tests (solution Mehdi + Oceane)")

Exemple guide : Stack fuzz tests (solution Mehdi + Oceane)


### Analyse : Fuzz tests pour une pile LIFO

**Solution validée** (Mehdi Robardet, Oceane Xiang, TP 2026) : Les fuzz tests vérifient les deux invariants fondamentaux d'une pile LIFO.

| Test | Invariant | Méthode |
|------|-----------|---------|
| `testFuzz_PushPop` | push+pop conserve la taille | Vérifie size avant/après + valeur retournée |
| `testFuzz_LIFO` | Dernier empilé = premier dépilé | Boucle inverse `i--` pour vérifier l'ordre |

**Points clés** :
- `vm.assume(values.length > 0 && values.length <= 100)` borne la taille du tableau (sinon out-of-gas)
- La boucle `for (uint i = values.length; i > 0; i--)` parcourt à l'envers pour vérifier LIFO
- `assertEq(popped, value)` dans PushPop vérifie que pop retourne exactement la valeur pushée
- L'approche stateless (setUp recrée la pile) garantit l'indépendance des tests

### Exercice : Fuzz tests pour une Queue FIFO

Créez des fuzz tests pour vérifier les propriétés d'une file d'attente (Queue FIFO).

In [10]:
# Exercice : Fuzz tests pour une Queue FIFO
# TODO etudiant : ecrire des fuzz tests pour verifier les proprietes d'une queue FIFO
# Indice : inspirez-vous des tests Stack LIFO (Exemple guide ci-dessus)
# Etape 1 : implementer testFuzz_EnqueueDequeue - verifier que enqueue puis dequeue laisse la taille inchangee
# Etape 2 : implementer testFuzz_FIFO - verifier que le premier element enfile est le premier defile

EXERCISE_QUEUE = '''
// Contrat a tester
contract Queue {
    uint256[] private items;
    uint256 private head;

    function enqueue(uint256 item) public {
        items.push(item);
    }
    
    function dequeue() public returns (uint256) {
        require(head < items.length, "Empty queue");
        uint256 value = items[head];
        head++;
        return value;
    }
    
    function size() public view returns (uint256) {
        return items.length - head;
    }
}

// Fuzz test
contract QueueFuzzTest is Test {
    Queue queue;
    
    function setUp() public {
        queue = new Queue();
    }
    
    // TODO etudiant : invariant enqueue+dequeue conserve la taille
    function testFuzz_EnqueueDequeue(uint256 value) public {
        // TODO etudiant : verifier que enqueue(value) incremente size de 1
        // TODO etudiant : verifier que dequeue() retourne value
        // TODO etudiant : verifier que size revient a la valeur initiale
        pass;
    }
    
    // TODO etudiant : invariant FIFO (first in, first out)
    function testFuzz_FIFO(uint256[] memory values) public {
        // TODO etudiant : filtrer avec vm.assume (length > 0, <= 100)
        // TODO etudiant : enqueue tous les elements
        // TODO etudiant : dequeue et verifier FIFO (ordre identique a l'insertion)
        // Indice : contrairement a LIFO, l'ordre de dequeue = l'ordre d'insertion
        pass;
    }
}
'''
print("Exercice a completer : Queue FIFO fuzz tests")

Exercice a completer : Queue FIFO fuzz tests


---

## 7. Resume

| Concept | Description |
|---------|-------------|
| Fuzz test | Test avec parametres aleatoires |
| `vm.assume` | Filtrer les entrees non desirees |
| Invariant | Propriete toujours vraie |
| `runs` | Nombre d'executions par test |

### Bonnes pratiques
1. Toujours limiter les ranges avec `vm.assume`
2. Tester les edge cases separement
3. Utiliser des seeds pour reproduire les bugs
4. Combiner fuzz tests et tests unitaires

---

**Notebook suivant** : [SC-14-Formal-Verification](SC-14-Formal-Verification.ipynb)

## Resume et perspectives

Ce notebook a permis de maitriser les techniques avancees de fuzz testing avec Foundry. Nous avons d'abord explore la generation d'inputs aleatoires pour decouvrir des edge cases invisibles dans les tests unitaires classiques, en appliquant les fonctions `testFuzz_*` sur des operations arithmetiques securisees (`safeAdd`, `safeMul`). Le filtrage d'inputs avec `vm.assume` a ete demontre comme un outil essentiel pour concentrer le fuzzing sur les domaines pertinents (exclusion de zero, bornes uint128, adresses EOA uniquement). Enfin, les tests d'invariants ont illustre comment verifier des proprietes structurelles comme la conservation du total supply d'un token ERC-20 ou la coherence des balances.

La configuration `foundry.toml` (parametres `runs`, `depth`, `seed`, `max_test_rejects`) offre un controle fin entre couverture et temps d'execution. En production, les contrats critiques sont generalement testes avec des milliers de runs et des seeds fixes pour garantir la reproductibilite des regressions detectees.

Le notebook suivant, [SC-14-Formal-Verification](SC-14-Formal-Verification.ipynb), va au-dela du fuzz testing en introduisant la verification formelle des smart contracts, une approche mathematique qui prouve l'absence de vulnerabilites plutot que de tenter de les decouvrir empiriquement.

---

[<< Precedent : Foundry Testing](SC-12-Foundry-Testing.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Formal Verification >>](SC-14-Formal-Verification.ipynb)

### Interpretation : Configuration Fuzz Foundry

La configuration trouvee definit comment Foundry execute les tests de fuzzing et d'invariants.

| Parametre | Valeur | Signification |
|-----------|--------|---------------|
| `runs = 1000` | Nombre d'iterations | Chaque test fuzz est execute 1000 fois avec des inputs differents |
| `max_test_rejects = 1000` | Limite de rejets | Si `vm.assume` rejette plus de 1000 inputs, le test est abandonne |
| `seed = '0x1234'` | Graine aleatoire | Permet de reproduire exactement le meme fuzz test (reproductibilite des bugs) |
| `depth = 15` | Profondeur invariants | Pour les invariants, nombre d'appels de fonctions aleatoires |
| `fail_on_revert = true` | Comportement revert | Un invariant echoue si une fonction revert |

**Points cles** :
1. `runs = 1000` est un bon compromis entre couverture et temps d'execution (defaut 256)
2. `seed` est critique pour le debugging : permet de rejouer exactement le meme cas qui a revele un bug
3. `max_test_rejects` evite les boucles infinies quand les contraintes `vm.assume` sont trop restrictives
4. La section `[invariant]` est specifique aux tests de proprietes (differents des tests fuzz simples)

> **Note technique** : En production, on utilise souvent `runs = 10000` ou plus pour les contrats critiques. Le `--fuzz-runs` en ligne de commande surcharge la config.
